# I. Initial Exploration of the Dataset

This notebook loads the cleaned marketing campaign data and performs basic data profiling.
The goal is to validate dataset structure, summary statistics, and missing-value patterns before deeper analysis.

In [15]:
from pathlib import Path
import shutil

# Set to True if you want to re-download even when the raw file already exists.
FORCE_DOWNLOAD = False
DATASET_DIR = Path("../dataset")
RAW_DATASET_PATH = DATASET_DIR / "marketing_campaign_dataset.csv"

DATASET_DIR.mkdir(parents=True, exist_ok=True)

if RAW_DATASET_PATH.exists() and not FORCE_DOWNLOAD:
    print(f"Raw dataset already available at: {RAW_DATASET_PATH.resolve()}")
else:
    try:
        import kagglehub
    except ImportError as exc:
        raise ImportError("kagglehub is required. Install with: pip install kagglehub") from exc

    kaggle_path = Path(
        kagglehub.dataset_download("manishabhatt22/marketing-campaign-performance-dataset")
    )

    source_csv = kaggle_path / "marketing_campaign_dataset.csv"
    if not source_csv.exists():
        csv_candidates = sorted(kaggle_path.rglob("*.csv"))
        if not csv_candidates:
            raise FileNotFoundError(f"No CSV files found in Kaggle download path: {kaggle_path}")
        source_csv = csv_candidates[0]

    shutil.copy2(source_csv, RAW_DATASET_PATH)
    print(f"Raw dataset copied to: {RAW_DATASET_PATH.resolve()}")

Raw dataset already available at: D:\VSCode\py_marketing_eda\dataset\marketing_campaign_dataset.csv


## Step 1: Acquire Raw Data

This block ensures the raw dataset is available in `../dataset`.

- If the file already exists, it reuses it for faster reruns.
- If not, it downloads from Kaggle and copies the CSV into the project dataset folder.

In [16]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Resolve project root whether kernel starts in /notebooks or repository root.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "scripts").exists() else cwd.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from scripts.data_cleaner import OUTPUT_CSV_FILENAME, clean_marketing_data

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

## Step 2: Configure Environment and Imports

This block imports required libraries, resolves the project root path, and performs the cleaning function from `scripts/data_cleaner.py`.

Display settings are also configured for easier dataframe inspection.

In [23]:
raw_file_path = RAW_DATASET_PATH

df = clean_marketing_data(raw_file_path)

print(f"DataFrame shape: {df.shape}")
print(f"Cleaned CSV saved to: {(Path('../dataset') / OUTPUT_CSV_FILENAME).resolve()}")

DataFrame shape: (200000, 16)
Cleaned CSV saved to: D:\VSCode\py_marketing_eda\dataset\marketing_campaign_dataset_cleaned.csv


## Step 3: Run Data Cleaning Pipeline

Here we execute `clean_marketing_data()` on the raw CSV and load the cleaned output into `df` for profiling.

The printed messages confirm row/column shape and cleaned file location.

In [18]:
df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Campaign_ID       200000 non-null  int64         
 1   Company           200000 non-null  object        
 2   Campaign_Type     200000 non-null  object        
 3   Target_Audience   200000 non-null  object        
 4   Duration          200000 non-null  Int64         
 5   Channel_Used      200000 non-null  object        
 6   Conversion_Rate   200000 non-null  float64       
 7   Acquisition_Cost  200000 non-null  float64       
 8   ROI               200000 non-null  float64       
 9   Location          200000 non-null  object        
 10  Language          200000 non-null  object        
 11  Clicks            200000 non-null  int64         
 12  Impressions       200000 non-null  int64         
 13  Engagement_Score  200000 non-null  int64         
 14  Cust

## Step 4: Inspect Schema and Data Types

Use `.info()` to verify column types, non-null counts, and memory footprint.

This helps confirm that cleaning converted `Date`, `Duration`, and `Acquisition_Cost` as expected.

In [19]:
numeric_profile = df.describe(include=[np.number]).T
numeric_profile = numeric_profile.sort_values(by="std", ascending=False)
numeric_profile.head(15)

,count,mean,std,min,25%,50%,75%,max
Campaign_ID,200000.0,100000.5,57735.171256,1.0,50000.75,100000.5,150000.25,200000.0
Acquisition_Cost,200000.0,12504.39304,4337.664545,5000.0,8739.75,12496.5,16264.0,20000.0
Impressions,200000.0,5507.30152,2596.864286,1000.0,3266.0,5517.5,7753.0,10000.0
Clicks,200000.0,549.77203,260.019056,100.0,325.0,550.0,775.0,1000.0
Duration,200000.0,37.503975,16.74672,15.0,30.0,30.0,45.0,60.0
Engagement_Score,200000.0,5.49471,2.872581,1.0,3.0,5.0,8.0,10.0
ROI,200000.0,5.002438,1.734488,2.0,3.5,5.01,6.51,8.0
Conversion_Rate,200000.0,0.08007,0.040602,0.01,0.05,0.08,0.12,0.15


## Step 5: Profile Numeric Distributions

This section summarizes numeric columns and sorts by standard deviation.

High-variance columns are useful starting points for deeper analysis and segmentation.

In [20]:
df.head()

,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date
0,1,Innovate Industries,Email,Men 18-24,30,Google Ads,0.04,16174.0,6.29,Chicago,Spanish,506,1922,6,Health & Wellness,2021-01-01
1,2,NexGen Systems,Email,Women 35-44,60,Google Ads,0.12,11566.0,5.61,New York,German,116,7523,7,Fashionistas,2021-01-02
2,3,Alpha Innovations,Influencer,Men 25-34,30,YouTube,0.07,10200.0,7.18,Los Angeles,French,584,7698,1,Outdoor Adventurers,2021-01-03
3,4,DataTech Solutions,Display,All Ages,60,YouTube,0.11,12724.0,5.55,Miami,Mandarin,217,1820,7,Health & Wellness,2021-01-04
4,5,NexGen Systems,Email,Men 25-34,15,YouTube,0.05,16452.0,6.50,Los Angeles,Mandarin,379,4201,3,Health & Wellness,2021-01-05


## Step 6: Preview Records and Check Completeness

`df.head()` provides a quick sanity check on cleaned records.

The missing-values summary reports both counts and percentages per column to prioritize data quality actions.

In [21]:
missing_count = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)

missing_summary = (
    pd.DataFrame({"missing_count": missing_count, "missing_pct": missing_pct})
    .sort_values(by=["missing_count", "missing_pct"], ascending=False)
)

print("Missing values summary per column:")
missing_summary

Missing values summary per column:


,missing_count,missing_pct
Campaign_ID,0,0.0
Company,0,0.0
Campaign_Type,0,0.0
Target_Audience,0,0.0
Duration,0,0.0
Channel_Used,0,0.0
Conversion_Rate,0,0.0
Acquisition_Cost,0,0.0
ROI,0,0.0
Location,0,0.0


## Summary: ROI and Conversion_Rate Distributions

Use the numeric profiling output to evaluate:

- `mean` vs `50%` (median): large gaps indicate skewness.
- `std` and interquartile range (`75% - 25%`): higher values indicate wider dispersion.
- `min` and `max`: extreme values can suggest outliers or long tails.

Interpretation checklist for this dataset:

- If **ROI** shows high spread and extreme upper values, campaign returns are likely heterogeneous.
- If **Conversion_Rate** has a tighter IQR and lower standard deviation, conversion outcomes are more stable across campaigns.
- If either metric has large skew, consider robust statistics (median, IQR) in later analysis.